In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round, sum, avg, split
import dlt

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("HealthcareDeltaLiveTable") \
    .getOrCreate()

# Task 1: Set up Delta Live Tables for real-time healthcare data ingestion

# Define the schema for the CSV or streaming data
schema = """
    patient_id INT, heart_rate INT, blood_pressure STRING,
    temperature DECIMAL(4,2), reading_time TIMESTAMP
"""

# Ingest raw vitals data into a Delta Live Table from the source (streaming CSV or medical devices)
@dlt.table(
  name="raw_vitals",
  comment="Raw patient vitals data ingested from devices."
)
def raw_vitals():
    return spark.readStream.format("csv").schema(schema).load("dbfs:/FileStore/vitals_fact.csv")

# Task 2: Data Processing with Delta Live Tables

# Clean the raw vitals data to filter out invalid entries (e.g., heart rate < 30 or temperature > 40)
@dlt.table(
  name="cleaned_vitals",
  comment="Cleaned vitals data with valid readings."
)
def cleaned_vitals():
    return dlt.read("raw_vitals").filter(
        (col("heart_rate") > 30) & (col("temperature") < 40)
    )

# Add transformations to process and enhance the healthcare data

# 1. Extract systolic and diastolic blood pressure from the blood_pressure string
@dlt.table(
  name="processed_vitals",
  comment="Vitals data with systolic and diastolic blood pressure extracted."
)
def processed_vitals():
    return dlt.read("cleaned_vitals") \
              .withColumn("systolic_bp", split(col("blood_pressure"), '/').getItem(0).cast("int")) \
              .withColumn("diastolic_bp", split(col("blood_pressure"), '/').getItem(1).cast("int"))

# 2. Calculate the average heart rate for each patient
@dlt.table(
  name="average_heart_rate",
  comment="Average heart rate per patient."
)
def average_heart_rate():
    return dlt.read("processed_vitals") \
              .groupBy("patient_id") \
              .agg(round(avg(col("heart_rate")), 2).alias("avg_heart_rate"))

# 3. Calculate the total number of high blood pressure cases per patient (systolic > 140)
@dlt.table(
  name="high_blood_pressure_cases",
  comment="Total number of high blood pressure cases per patient."
)
def high_blood_pressure_cases():
    return dlt.read("processed_vitals") \
              .filter(col("systolic_bp") > 140) \
              .groupBy("patient_id") \
              .agg(sum(col("systolic_bp") > 140).alias("high_bp_cases"))

# 4. Calculate daily average temperature across all patients
@dlt.table(
  name="daily_avg_temperature",
  comment="Daily average temperature across all patients."
)
def daily_avg_temperature():
    return dlt.read("processed_vitals") \
              .groupBy("reading_time") \
              .agg(round(avg(col("temperature")), 2).alias("daily_avg_temperature"))
